# Digital-Twin Gap Study — Colab / Kaggle Runner

This notebook builds **digital twins of shopping decisions** with three different
architectures and measures the **gap between how the twin performs on LLM *agents*
vs on real *humans***.

**Three architectures (all predict which option A/B/C/D is chosen):**
1. `tfidf_logreg` — TF-IDF + Logistic Regression (classic ML floor, CPU, seconds)
2. `embed_mlp` — frozen MiniLM sentence embeddings + a small MLP head
3. `distilbert` — fine-tuned DistilBERT sequence classifier (needs a GPU to be quick)

**The experiment:** for each architecture we run the full 2x2 transfer matrix
— train on agent / human x test on agent / human — and report the **gap**
(matched-population accuracy minus transferred-population accuracy).

> Runs on **Google Colab**, **Kaggle**, or **locally**. Run the cells top to bottom.
> For `distilbert`, enable a **GPU** runtime (Runtime -> Change runtime type -> GPU).


## 1. Get the project code + data

Pick **one** of the options below depending on where your files are.

### Option A - Upload the project zip (works everywhere)

In [ ]:
# --- Detect environment ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE)


In [ ]:
# --- Option A: upload a zip of the project (contains src/, configs/, data/raw/) ---
# Skip this cell if using Option B (Drive) or Option C (Kaggle dataset).
if IN_COLAB:
    from google.colab import files
    print("Select digital_twin_gap.zip ...")
    up = files.upload()
    zip_name = list(up.keys())[0]
    get_ipython().system('unzip -o -q "$zip_name" -d /content/')
    for root, dirs, fnames in os.walk("/content"):
        if "src" in dirs and os.path.exists(os.path.join(root, "src", "run_all.py")):
            PROJECT_ROOT = root
            break
    print("PROJECT_ROOT =", PROJECT_ROOT)


### Option B - Mount Google Drive (Colab)

In [ ]:
# --- Option B: Google Drive (Colab only) ---
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/digital_twin_gap"   # EDIT to your path
    print("PROJECT_ROOT =", PROJECT_ROOT)


### Option C - Kaggle dataset (Add Input, then set the path)

In [ ]:
# --- Option C: Kaggle dataset ---
if IN_KAGGLE:
    src_dir = "/kaggle/input/digital-twin-gap"        # EDIT to your dataset folder
    PROJECT_ROOT = "/kaggle/working/digital_twin_gap"
    get_ipython().system('mkdir -p "$PROJECT_ROOT"')
    get_ipython().system('cp -r "$src_dir"/* "$PROJECT_ROOT"/')
    print("PROJECT_ROOT =", PROJECT_ROOT)


### Local fallback

In [ ]:
# --- Local fallback (only if PROJECT_ROOT wasn't set above) ---
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.getcwd()
os.chdir(PROJECT_ROOT)
print("Now in:", os.getcwd())
print("Contents:", os.listdir("."))


## 2. Install dependencies

In [ ]:
get_ipython().system('pip install -q -r requirements.txt')
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Point the config at your data (only if needed)

`configs/config.yaml` expects:
- Agentic `.xlsx` under `data/raw/Agentic response/` (Claude/, Chatgpt/, Grok 4.2/)
- Human `.jsonl` under `data/raw/human/` (`llm_train_human.jsonl`, `llm_val_human.jsonl`)

In [ ]:
import yaml
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# EDIT these only if your data is elsewhere:
# cfg["data"]["agentic_root"] = "data/raw/Agentic response"
# cfg["data"]["human_files"]  = ["data/raw/human/llm_train_human.jsonl",
#                                "data/raw/human/llm_val_human.jsonl"]

with open("configs/config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("agentic_root:", cfg["data"]["agentic_root"], "->", os.path.exists(cfg["data"]["agentic_root"]))
for hf in cfg["data"]["human_files"]:
    print("human file:", hf, "->", os.path.exists(hf))


## 4. Build the unified dataset

In [ ]:
get_ipython().system("python -m src.data_prep --config configs/config.yaml")

## 5. Run the experiments

`distilbert` is the slow one - enable a GPU runtime, or skip it.

In [ ]:
# Quick: classic ML only (fast, no GPU)
get_ipython().system("python -m src.run_all --config configs/config.yaml --models tfidf_logreg")

In [ ]:
# Full run: all three (GPU recommended for distilbert)
get_ipython().system("python -m src.run_all --config configs/config.yaml --models tfidf_logreg embed_mlp distilbert")

## 6. Analyze + plot the gap

In [ ]:
get_ipython().system("python -m src.analyze --config configs/config.yaml")

In [ ]:
from IPython.display import Image, display
for fig in ["fig_transfer_matrix.png", "fig_gap_bars.png", "fig_per_agent.png"]:
    p = os.path.join("results", fig)
    if os.path.exists(p):
        print(fig); display(Image(p))


In [ ]:
import pandas as pd
print("=== GAP REPORT ==="); display(pd.read_csv("results/gap_report.csv"))
print("=== FULL MATRIX ==="); display(pd.read_csv("results/summary_matrix.csv"))


## 7. Download the results

In [ ]:
get_ipython().system("zip -r -q results.zip results/")
if IN_COLAB:
    from google.colab import files
    files.download("results.zip")
elif IN_KAGGLE:
    print("results.zip is in /kaggle/working/ - grab it from the Output tab.")
else:
    print("results.zip at", os.path.abspath("results.zip"))
